# 1. functions as parameters

In Python, you can use a function wherever you are expecting a value. So e.g. as value of parameters in a function call. In the example below you see the 'strange' `lambda`-syntax (which Guido van Rossum actually [didn't want to have in Python](https://developers.slashdot.org/story/13/08/25/2115204/interviews-guido-van-rossum-answers-your-questions)).

In [ ]:
def demo_functions_parameters(vals, cb):
    return [cb(x) for x in vals]

l = [2,3,5,7,11,13]
print(demo_functions_parameters(l, lambda x: x**2)) #anonymous function
print(demo_functions_parameters(l, lambda x: x**3))

In [ ]:
def square(v):
    return v*v

print(demo_functions_parameters(l, square)) #anonymous function


You can also create a function that returns a function, so that you actually don't need a `lamdba`-syntax:

In [ ]:
def make_adder(n):
    def adder(x):
        return x+n
    
    return adder

a = make_adder(4)
print (a(5))

In [ ]:
import math

def demo_function(x):
    return math.sqrt(x)

print(demo_functions_parameters(l, demo_function))

## 2. Generators

Generators in Python are functions that produce a sequence of values lazily, allowing for efficient memory usage and on-the-fly generation of values. The interesting thing about generators is that they can exit while keeping their state...

In [ ]:
def get_id():
    ctr = 1
    while True:
        yield ctr
        ctr += 1

id_generator = get_id()
print (type(id_generator))
for i in range(10):
    id = next(id_generator)
    print(f'User with id {id}')

In [ ]:
next(id_generator)

# 3. Fetching data from a backend

We can simpy use the library [`requests`](https://requests.readthedocs.io/en/latest/) to fetch data from another server:

In [ ]:
import requests
response = requests.get('https://jsonplaceholder.typicode.com/posts')

if response.status_code == 200:
    data = response.json()
    print(data[0])
else:
    print('Error fetching data')

Sometimes however, this getting of the data can take a while, e.g. when the server or the network is very slow, or if there's a lot of data to ge. In such case, we can use a technique similar to generators to have the main program continue and have the control given back to the function that takes long as soon as it is finished (and there's nothing else going on.

Such a function is called *asynchronous* and we define it using the `async` keyword. There are a few methods to accomplish this in Python (Python being Python), but we'll demonstrate the use of [`aiohttp`](https://pypi.org/project/aiohttp/) and [`asyncio`](https://docs.python.org/3/library/asyncio.html).

# 4. aiohttp and asyncio

You use the library `asyncio` to write concurrent code using the `async/await` syntax. It is used as a foundation for multiple Python asynchronous frameworks that provide high-performance network and web-servers, database connection libraries, distributed task queues, etc.

Have a look at the code below. Here, we use this library to fetch data from a remote server. If you try to run this within the Notebook, however, you will receive an error: `RuntimeError: asyncio.run() cannot be called from a running event loop`.

In [ ]:
import aiohttp
import asyncio
import json

async def fetch_json(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
            data = await response.json()
            return data

async def main():
    url = 'http://jsonplaceholder.typicode.com/posts/1'
    data = await fetch_json(url)
    print(json.dumps(data, indent=4))

asyncio.run(main())

When running asyncio code in a Jupyter notebook or any other interactive environment, you need to warp the call to `asyncio.run()` within another asynchronous function due to how event loops work in such environments.

In [ ]:
async def f_one():
    print("Hello")
    await asyncio.sleep(1)
    print("World! 👋")

async def f_two():
    print("    One")
    await asyncio.sleep(0.5)
    print("    Two")
    await asyncio.sleep(0.5)
    print("    Three")

async def main():
    await asyncio.gather(f_one(), f_two())

async def run_main():
    await main()

await run_main()


So our code to fetch data from the remote server becomes somewhat more bloated:

In [ ]:
async def fetch_json(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
            data = await response.json()
            return data

async def get_data():
    url = 'http://jsonplaceholder.typicode.com/posts/1'
    data = await fetch_json(url)
    print(json.dumps(data, indent=4))

async def run_fetch():
    await get_data()
    
await run_fetch()

# 4. Event driven programming

In Python, while there's no built-in concept of events like in JavaScript, you can implement a similar pattern using classes and callbacks. You can create your own event system by defining a class to represent an event and allowing other parts of your code to subscribe to or listen for these events.

In [ ]:
class Event:
    def __init__(self):
        self.handlers = []

    def add_handler(self, handler):
        self.handlers.append(handler)

    def remove_handler(self, handler):
        self.handlers.remove(handler)

    def fire(self, *args, **kwargs):
        for handler in self.handlers:
            handler(*args, **kwargs)

# event listeners
def on_event_fired(data):
    print(f"Event fired with data: {data}")

def another_listener(data):
    print('another listerener was triggered...')
    
custom_event = Event()
custom_event.add_handler(on_event_fired)
custom_event.fire("Demonstration data")

custom_event.remove_handler(on_event_fired)


# 5. Multithreading

Multithreading is a programming technique that allows multiple threads to execute concurrently within a single process, enabling applications to perform multiple tasks simultaneously and improve responsiveness by utilizing the available CPU resources more efficiently.

In [ ]:
import threading
import time

def print_numbers():
    for i in range(5):
        print(i)
        time.sleep(1)

def print_letters():
    for letter in 'abcde':
        print(letter)
        time.sleep(1)

# Create two threads
thread1 = threading.Thread(target=print_numbers)
thread2 = threading.Thread(target=print_letters)

# Start the threads
thread1.start()
thread2.start()
print('This line is executed while the threads keep running...')

# Wait for both threads to finish
thread1.join()
thread2.join()
print('all threads are finished')

